# Diffusion-population diagnostics

This final diagnostic examines the Stage 5 connected diffusion population before any diffusion model is implemented. It preserves every selected user, including zero-out-degree receivers, and uses the graph as a static directed weighted network.

## Load Stage 5 population

In [ ]:
from pathlib import Path
import json
import math

import networkx as nx
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

candidates=[Path.cwd().resolve(),Path.cwd().resolve().parent]
PROJECT_ROOT=next((p for p in candidates if (p/'data'/'processed'/'diffusion_population_users.csv').exists()),None)
if PROJECT_ROOT is None: raise FileNotFoundError('Run after Stage 5 outputs have been generated.')
PROCESSED_DIR=PROJECT_ROOT/'data'/'processed'; TABLES_DIR=PROJECT_ROOT/'results'/'tables'; FIGURES_DIR=PROJECT_ROOT/'results'/'figures'
for directory in [PROCESSED_DIR,TABLES_DIR,FIGURES_DIR]: directory.mkdir(parents=True,exist_ok=True)

users=pd.read_csv(PROCESSED_DIR/'diffusion_population_users.csv')
edges=pd.read_csv(PROCESSED_DIR/'diffusion_population_edges.csv')
tweets=pd.read_csv(PROCESSED_DIR/'diffusion_population_tweets.csv')
print(f'Loaded {len(users):,} users, {len(edges):,} directed edges, and {len(tweets):,} tweets')

def save_bar_chart(labels,values,title,filename,color=(55,126,184)):
    width,height,margin=1200,680,85
    image=Image.new('RGB',(width,height),'white'); draw,font=ImageDraw.Draw(image),ImageFont.load_default()
    maximum=max(values) if values else 1; slot=(width-2*margin)/max(len(values),1); bar_width=max(2,int(slot*.70))
    draw.text((margin,25),title,fill='black',font=font); draw.line((margin,height-margin,width-margin,height-margin),fill='black',width=2); draw.line((margin,margin,margin,height-margin),fill='black',width=2)
    for i,(label,value) in enumerate(zip(labels,values)):
        x0=int(margin+i*slot+(slot-bar_width)/2); y0=int(height-margin-(value/maximum)*(height-2*margin-45)) if maximum else height-margin
        draw.rectangle((x0,y0,x0+bar_width,height-margin),fill=color); draw.text((x0,max(margin+22,y0-15)),f'{value:.3g}',fill='black',font=font)
        if len(values)<=25: draw.text((x0,height-margin+8),str(label)[:18],fill='black',font=font)
    output=FIGURES_DIR/filename; image.save(output); print(f'Saved figure: {output.relative_to(PROJECT_ROOT)}')

def save_histogram(values,title,filename,bins=40):
    series=pd.Series(values).dropna().astype(float).map(math.log1p); low,high=series.min(),series.max(); step=(high-low)/bins if high!=low else 1
    counts=[0]*bins
    for value in series: counts[min(int((value-low)/step),bins-1)]+=1
    save_bar_chart([f'{low+i*step:.2f}' for i in range(bins)],counts,title,filename,color=(77,175,74))


## Part 1 — Out-degree analysis

In [ ]:
# Recompute directed degree metrics from the clean Stage 5 edge table.
G=nx.DiGraph(); G.add_nodes_from(users['node_id'].astype(str)); G.add_weighted_edges_from(edges[['Source','Target','Weight']].itertuples(index=False,name=None),weight='Weight')
diagnostic_nodes=pd.DataFrame({'node_id':list(G.nodes())})
diagnostic_nodes['in_degree']=diagnostic_nodes['node_id'].map(dict(G.in_degree())).astype(int)
diagnostic_nodes['out_degree']=diagnostic_nodes['node_id'].map(dict(G.out_degree())).astype(int)
diagnostic_nodes['total_degree']=diagnostic_nodes['in_degree']+diagnostic_nodes['out_degree']
diagnostic_nodes['weighted_in_degree']=diagnostic_nodes['node_id'].map(dict(G.in_degree(weight='Weight'))).astype(float)
diagnostic_nodes['weighted_out_degree']=diagnostic_nodes['node_id'].map(dict(G.out_degree(weight='Weight'))).astype(float)
diagnostic_nodes['weighted_degree']=diagnostic_nodes['weighted_in_degree']+diagnostic_nodes['weighted_out_degree']
degree_statistics=diagnostic_nodes[['in_degree','out_degree','total_degree','weighted_in_degree','weighted_out_degree','weighted_degree']].describe(percentiles=[.25,.5,.75]).T[['mean','50%','std','min','25%','75%','max']]
degree_statistics=degree_statistics.rename(columns={'50%':'median','25%':'p25','75%':'p75'})
print(degree_statistics.to_string())
out_degree_counts={'out_degree_0':int((diagnostic_nodes.out_degree==0).sum()),'out_degree_1':int((diagnostic_nodes.out_degree==1).sum()),'out_degree_gte_2':int((diagnostic_nodes.out_degree>=2).sum()),'out_degree_gte_5':int((diagnostic_nodes.out_degree>=5).sum()),'out_degree_gte_10':int((diagnostic_nodes.out_degree>=10).sum()),'out_degree_gte_20':int((diagnostic_nodes.out_degree>=20).sum())}
print(json.dumps(out_degree_counts,indent=2))
save_histogram(diagnostic_nodes['out_degree'],'Out-degree distribution (log1p degree)','out_degree_distribution.png')
save_histogram(diagnostic_nodes['weighted_out_degree'],'Weighted out-degree distribution (log1p degree)','weighted_out_degree_distribution.png')

## Part 2 — Spreading roles

In [ ]:
# These are descriptive analytical roles only; no user is removed or altered.
def spreading_role(row):
    if row.in_degree==0 and row.out_degree==0: return 'isolated'
    if row.in_degree>0 and row.out_degree==0: return 'receiver_only'
    if row.out_degree>=2: return 'active_spreader'
    return 'spreader'
diagnostic_nodes['spreading_role']=diagnostic_nodes.apply(spreading_role,axis=1)
role_counts=diagnostic_nodes['spreading_role'].value_counts().reindex(['isolated','receiver_only','spreader','active_spreader'],fill_value=0)
role_summary=pd.DataFrame({'count':role_counts,'percentage':100*role_counts/len(diagnostic_nodes)})
print(role_summary.to_string(float_format=lambda value:f'{value:.2f}%'))

## Part 3 — Target information by spreading role

In [ ]:
content_columns=['node_id','tweet_count','negative_tweet_count','positive_tweet_count','negative_ratio','positive_ratio','dominant_target','avg_polarity','avg_subjectivity','pagerank','trust_score']
content=users[content_columns].rename(columns={'avg_polarity':'average_polarity','avg_subjectivity':'average_subjectivity'})
diagnostic_nodes=diagnostic_nodes.merge(content,on='node_id',how='left',validate='one_to_one')
assert diagnostic_nodes['tweet_count'].notna().all()
information_counts={
    'users_with_negative_information':int((diagnostic_nodes.negative_tweet_count>0).sum()),
    'users_with_positive_information':int((diagnostic_nodes.positive_tweet_count>0).sum()),
    'users_with_both':int(((diagnostic_nodes.negative_tweet_count>0)&(diagnostic_nodes.positive_tweet_count>0)).sum()),
    'users_only_negative':int(((diagnostic_nodes.negative_tweet_count>0)&(diagnostic_nodes.positive_tweet_count==0)).sum()),
    'users_only_positive':int(((diagnostic_nodes.negative_tweet_count==0)&(diagnostic_nodes.positive_tweet_count>0)).sum())
}
print(json.dumps(information_counts,indent=2))
role_target_crosstab=pd.crosstab(diagnostic_nodes['spreading_role'],diagnostic_nodes['dominant_target']).reindex(index=['isolated','receiver_only','spreader','active_spreader'],fill_value=0)
print('\nSpreading role by dominant Target (descriptive only):\n'+role_target_crosstab.to_string())
role_target_crosstab.to_csv(TABLES_DIR/'diffusion_population_role_target_crosstab.csv')

def save_grouped_bars(frame,title,filename):
    labels=frame.index.tolist(); columns=frame.columns.tolist(); width,height,margin=1200,680,85
    image=Image.new('RGB',(width,height),'white'); draw,font=ImageDraw.Draw(image),ImageFont.load_default(); maximum=frame.sum(axis=1).max() or 1
    draw.text((margin,25),title,fill='black',font=font); draw.line((margin,height-margin,width-margin,height-margin),fill='black',width=2); draw.line((margin,margin,margin,height-margin),fill='black',width=2)
    colors=[(55,126,184),(255,127,0),(152,78,163)]; slot=(width-2*margin)/max(len(labels),1); bar_width=max(2,int(slot*.22))
    for i,label in enumerate(labels):
        center=margin+i*slot+slot/2
        for j,column in enumerate(columns):
            value=frame.loc[label,column]; x0=int(center+(j-len(columns)/2)*bar_width); y0=int(height-margin-(value/maximum)*(height-2*margin-45))
            draw.rectangle((x0,y0,x0+bar_width,height-margin),fill=colors[j%len(colors)])
        draw.text((int(center-slot/3),height-margin+8),str(label),fill='black',font=font)
    draw.text((width-360,25),' / '.join(map(str,columns)),fill='black',font=font); output=FIGURES_DIR/filename; image.save(output); print(f'Saved figure: {output.relative_to(PROJECT_ROOT)}')

save_grouped_bars(role_target_crosstab,'Dominant Target by spreading role','target_by_spreading_role.png')

## Part 4 — Trust and spreading relationships

In [ ]:
relationship_columns=['trust_score','pagerank','weighted_degree','out_degree']
pearson=diagnostic_nodes[relationship_columns].corr(method='pearson')
spearman=diagnostic_nodes[relationship_columns].rank(method='average').corr(method='pearson')
print('Pearson correlations:\n'+pearson.to_string())
print('\nSpearman correlations:\n'+spearman.to_string())
correlation_output=pd.concat({'pearson':pearson,'spearman':spearman},axis=0)
correlation_output.to_csv(TABLES_DIR/'diffusion_population_diagnostic_correlations.csv')

def save_scatter(x,y,title,filename):
    width,height,margin=1200,680,85
    image=Image.new('RGB',(width,height),'white'); draw,font=ImageDraw.Draw(image),ImageFont.load_default()
    x,y=pd.Series(x).astype(float),pd.Series(y).astype(float); x_low,x_high=x.min(),x.max(); y_low,y_high=y.min(),y.max(); x_range=x_high-x_low or 1; y_range=y_high-y_low or 1
    draw.text((margin,25),title,fill='black',font=font); draw.line((margin,height-margin,width-margin,height-margin),fill='black',width=2); draw.line((margin,margin,margin,height-margin),fill='black',width=2)
    for xv,yv in zip(x,y):
        px=int(margin+(xv-x_low)/x_range*(width-2*margin)); py=int(height-margin-(yv-y_low)/y_range*(height-2*margin)); draw.ellipse((px-1,py-1,px+1,py+1),fill=(55,126,184))
    output=FIGURES_DIR/filename; image.save(output); print(f'Saved figure: {output.relative_to(PROJECT_ROOT)}')

save_scatter(diagnostic_nodes['trust_score'],diagnostic_nodes['out_degree'].map(math.log1p),'Trust score vs out-degree (log1p out-degree)','trust_vs_out_degree.png')
print('Correlations are descriptive associations and do not establish causal influence.')

## Part 5 — Diffusion readiness

In [ ]:
strong_components=list(nx.strongly_connected_components(G)); largest_scc=max(map(len,strong_components)); nontrivial_scc_nodes=sum(len(component) for component in strong_components if len(component)>1)
readiness={
    'potential_transmitters_out_degree_gt_0':int((diagnostic_nodes.out_degree>0).sum()),
    'receiver_only_users':int((diagnostic_nodes.spreading_role=='receiver_only').sum()),
    'both_information_types_present':bool(information_counts['users_with_negative_information'] and information_counts['users_with_positive_information']),
    'users_with_both_information_types':information_counts['users_with_both'],
    'directed_edges':G.number_of_edges(), 'weakly_connected':nx.is_weakly_connected(G), 'strongly_connected':nx.is_strongly_connected(G),
    'largest_strong_component_size':largest_scc, 'nontrivial_strong_component_nodes':nontrivial_scc_nodes,
    'directed_reachability_note':'Asymmetric: the graph is weakly but not strongly connected, so not every node can reach every other node along directed paths.'
}
print(json.dumps(readiness,indent=2))
print('Readiness conclusion: the population has substantial directed propagation structure and both information types, but receiver-only users and non-strong connectivity must be represented explicitly in later simulations.')

## Part 6 — Clean diffusion input tables

In [ ]:
model_node_columns=['node_id','in_degree','out_degree','total_degree','weighted_in_degree','weighted_out_degree','weighted_degree','pagerank','trust_score','tweet_count','negative_tweet_count','positive_tweet_count','negative_ratio','positive_ratio','dominant_target','average_polarity','average_subjectivity']
model_nodes=diagnostic_nodes[model_node_columns].sort_values('node_id').reset_index(drop=True)
model_edges=edges[['Source','Target','Weight']].copy()
assert len(model_nodes)==G.number_of_nodes() and len(model_edges)==G.number_of_edges()
model_nodes.to_csv(PROCESSED_DIR/'diffusion_model_nodes.csv',index=False)
model_edges.to_csv(PROCESSED_DIR/'diffusion_model_edges.csv',index=False)
print(f'Saved clean node table ({len(model_nodes):,} rows) and directed edge table ({len(model_edges):,} rows); no timestamps added.')

## Part 7 — Proposed Competitive Diffusion Model

In [ ]:
proposed_model_notes = '''
Proposed Competitive Diffusion Model

The next stage will use four node states:
S  = susceptible
I1 = positive-information infected
I2 = negative-information infected
R  = recovered

It will model competition between I1 and I2 over this directed social network. Transition probabilities will be defined later from edge weight, trust, sentiment, positive/negative information type, and network structure. No diffusion equations, numerical transmission probabilities, GNN training, or EvolveGCN implementation are introduced in this diagnostic stage.
'''.strip()
print(proposed_model_notes)
print('\nDiagnostic execution complete. The graph remains static because the edge file has no timestamps; monthly content features are separate temporal inputs.')